#### CSV PARSING AND VALIDATION

In [1]:
import csv
import pandas as pd

total_data_rows = 0
csv_valid_rows = []
csv_malformed_rows = []


with open("reviews.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)

    header = next(reader)

    for row in reader:
        total_data_rows += 1

        physical_end_line = reader.line_num

        embedded_newlines = sum(
            field.count("\n")
            for field in row
        )

        first_physical_line = (
            physical_end_line - embedded_newlines
        )

        if len(row) == 14:
            csv_valid_rows.append({
                "line_number": first_physical_line,
                "row": row
            })
        else:
            csv_malformed_rows.append({
                "line_number": first_physical_line,
                "field_count": len(row),
                "row": row
            })

print("Header fields:", len(header))
print("Total data rows:", total_data_rows)
print("CSV-valid rows:", len(csv_valid_rows))
print("CSV-malformed rows:", len(csv_malformed_rows))

Header fields: 14
Total data rows: 10548
CSV-valid rows: 10521
CSV-malformed rows: 27


##### There are 10,548 CSV records and 14 headers.
##### 27 CSV-rows are malformed.
##### Therefore, those 27 CSV-rows are malformed at the CSV-parsing level and must be discarded.

In [2]:
df = pd.DataFrame(
    [item["row"] for item in csv_valid_rows],
    columns=header
)

df["line_number"] = [
    item["line_number"]
    for item in csv_valid_rows
]

In [3]:
print("DataFrame shape:", df.shape)

DataFrame shape: (10521, 15)


In [4]:
df[["line_number", "submission_id", "reviewer_id"]].head()

,line_number,submission_id,reviewer_id
0,2,S02124,R0699
1,3,S00870,R0817
2,4,S02914,R0080
3,5,S00360,R0270
4,6,S00434,R0372


#### SCHEMA LEVEL VALIDATION

In [5]:
df.head()

,submission_id,paper_title,track,author_ids,reviewer_id,assignment_timestamp,review_timestamp,soundness,excitement,confidence,overall_score,metareview_score,review_text,recommendation,line_number
0,S02124,On the State-Space Models for Long-Context Rea...,MainConference,A3145|A1367|A1303,R0699,2025-02-28T18:08:22Z,2025-03-08T00:08:22Z,2,3,3,5.1,7.8,I appreciate the clear writing and structure. ...,Borderline,2
1,S00870,"Towards Retrieval-Augmented Methods, with Appl...",Findings,A0101|A1645|A2529,R0817,2025-03-30T06:08:07Z,2025-04-09T22:08:07Z,2,3,3,5.2,7.5,"The authors write, ""our method outperforms all...",Borderline,3
2,S02914,On the Contrastive Models for Long-Context Rea...,Findings,A1417|A2312|A0073,R0080,2025-03-15T11:11:52Z,2025-03-23T04:11:52Z,4,4,5,7.3,4.2,Ablations would strengthen the empirical story...,Accept,4
3,S00360,"Efficient Transformer Architectures, Revisited",MainConference,A0688|A3197|A1873,R0270,2025-03-28T00:47:38Z,2025-04-07T09:47:38Z,2,2,2,4.3,5.9,Scalability beyond the tested regime is unclea...,Reject,5
4,S00434,"Robust Diffusion Architectures, Revisited",MainConference,A3997|A0645,R0372,2025-03-17T01:19:20Z,2025-04-01T20:19:20Z,3,3,1,5.0,6.9,Related work coverage is adequate but could be...,Borderline,6


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10521 entries, 0 to 10520
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   submission_id         10521 non-null  object
 1   paper_title           10521 non-null  object
 2   track                 10521 non-null  object
 3   author_ids            10521 non-null  object
 4   reviewer_id           10521 non-null  object
 5   assignment_timestamp  10521 non-null  object
 6   review_timestamp      10521 non-null  object
 7   soundness             10521 non-null  object
 8   excitement            10521 non-null  object
 9   confidence            10521 non-null  object
 10  overall_score         10521 non-null  object
 11  metareview_score      10521 non-null  object
 12  review_text           10521 non-null  object
 13  recommendation        10521 non-null  object
 14  line_number           10521 non-null  int64 
dtypes: int64(1), object(14)
memory usage

In [7]:
df.dtypes

submission_id           object
paper_title             object
track                   object
author_ids              object
reviewer_id             object
assignment_timestamp    object
review_timestamp        object
soundness               object
excitement              object
confidence              object
overall_score           object
metareview_score        object
review_text             object
recommendation          object
line_number              int64
dtype: object

In [8]:
df.isnull().sum()

submission_id           0
paper_title             0
track                   0
author_ids              0
reviewer_id             0
assignment_timestamp    0
review_timestamp        0
soundness               0
excitement              0
confidence              0
overall_score           0
metareview_score        0
review_text             0
recommendation          0
line_number             0
dtype: int64

##### Validate submission_id and reviewer_id

In [9]:
valid_submission_id = (
    df["submission_id"].notna()
    & df["submission_id"].astype(str).str.strip().ne("")
)

valid_reviewer_id = (
    df["reviewer_id"].notna()
    & df["reviewer_id"].astype(str).str.strip().ne("")
)

##### Validate track and recommendation

In [10]:
allowed_tracks = {
    "MainConference",
    "Findings",
    "Workshop",
    "Demo"
}

allowed_recommendations = {
    "Strong Accept",
    "Accept",
    "Borderline",
    "Reject",
    "Strong Reject"
}

valid_track = df['track'].isin(allowed_tracks)
valid_recommendation = df["recommendation"].isin(
    allowed_recommendations
)

In [11]:
(~valid_track).sum()

0

In [12]:
df.loc[~valid_track, ["submission_id", "track"]]

,submission_id,track


In [13]:
(~valid_recommendation).sum()

0

In [14]:
df.loc[~valid_recommendation, ["submission_id", "recommendation"]].head(2)

,submission_id,recommendation


In [15]:
assignment_dt = pd.to_datetime(
    df["assignment_timestamp"],
    errors="coerce",
    utc=True
)

review_dt = pd.to_datetime(
    df["review_timestamp"],
    errors="coerce",
    utc=True
)

In [16]:
valid_assignment_timestamp = assignment_dt.notna()
valid_review_timestamp = review_dt.notna()

In [17]:
(~valid_assignment_timestamp).sum()

37

In [18]:
(~valid_review_timestamp).sum()

31

In [19]:
df.loc[~valid_assignment_timestamp, ["submission_id", "assignment_timestamp"]].head(2)

,submission_id,assignment_timestamp
327,S02374,2025/03/14
991,S00204,2025/03/14


In [20]:
df.loc[~valid_review_timestamp, ["submission_id", "review_timestamp"]].head(2)

,submission_id,review_timestamp
568,S01360,yesterday
609,S00895,yesterday


##### Validate soundness, excitement and confidence

In [21]:
def valid_integer_score(label, lower=1, upper=5):
    parsed = pd.to_numeric(label, errors="coerce")
    
    return (
        parsed.notna()
        & (parsed % 1 == 0)
        & parsed.between(lower, upper)
    )

In [22]:
valid_soundness = valid_integer_score(df["soundness"])
valid_excitement = valid_integer_score(df["excitement"])
valid_confidence = valid_integer_score(df["confidence"])

In [23]:
(~valid_soundness).sum()

32

In [24]:
(~valid_excitement).sum()

0

In [25]:
(~valid_confidence).sum()

38

In [26]:
df.loc[~valid_soundness, ["submission_id", "soundness"]].head(2)

,submission_id,soundness
259,S01847,high
534,S02590,high


In [27]:
df.loc[~valid_excitement, ["submission_id", "excitement"]]

,submission_id,excitement


In [28]:
df.loc[~valid_confidence, ["submission_id", "confidence"]].head(2)

,submission_id,confidence
440,S01087,99
638,S00157,99


##### Validate overall_score and metareview_score

In [29]:
def valid_float_score(label, lower=1.0, upper=10.0):
    parsed = pd.to_numeric(label, errors="coerce")
    
    return (
        parsed.notna()
        & parsed.between(lower, upper)
    )

In [30]:
valid_overall_score = valid_float_score(
    df["overall_score"]
)

valid_metareview_score = valid_float_score(
    df["metareview_score"]
)

In [31]:
(~valid_overall_score).sum()

65

In [32]:
(~valid_metareview_score).sum()

0

In [33]:
df.loc[~valid_overall_score, ["submission_id", "overall_score"]]

,submission_id,overall_score
178,S00638,N/A
374,S02132,N/A
484,S02105,49.3
600,S00105,N/A
918,S00351,N/A
...,...,...
9463,S00303,N/A
10075,S02046,N/A
10096,S00353,N/A
10109,S00938,N/A


In [34]:
df.loc[~valid_metareview_score, ["submission_id", "metareview_score"]].head(2)

,submission_id,metareview_score


#### Global Row Validation
A row is valid if and only if every condition below is true.

In [35]:
valid_row = (
    valid_submission_id
    & valid_reviewer_id
    & valid_track
    & valid_assignment_timestamp
    & valid_review_timestamp
    & valid_soundness
    & valid_excitement
    & valid_confidence
    & valid_overall_score
    & valid_metareview_score
    & valid_recommendation
)

In [36]:
valid_row.value_counts()

True     10273
False      248
Name: count, dtype: int64

In [37]:
valid_count = valid_row.sum()

valid_count

10273

In [38]:
malformed_count = (~valid_row).sum()

malformed_count

248

In [39]:
valid_count + malformed_count == len(df)

True

##### Inspect and assert malformed rows

In [40]:
validation = pd.DataFrame(index=df.index)

validation["invalid_submission_id"] = ~valid_submission_id
validation["invalid_reviewer_id"] = ~valid_reviewer_id
validation["invalid_track"] = ~valid_track
validation["invalid_assignment_timestamp"] = ~valid_assignment_timestamp
validation["invalid_review_timestamp"] = ~valid_review_timestamp
validation["invalid_soundness"] = ~valid_soundness
validation["invalid_excitement"] = ~valid_excitement
validation["invalid_confidence"] = ~valid_confidence
validation["invalid_overall_score"] = ~valid_overall_score
validation["invalid_metareview_score"] = ~valid_metareview_score
validation["invalid_recommendation"] = ~valid_recommendation

In [41]:
validation.sum()

invalid_submission_id           21
invalid_reviewer_id             24
invalid_track                    0
invalid_assignment_timestamp    37
invalid_review_timestamp        31
invalid_soundness               32
invalid_excitement               0
invalid_confidence              38
invalid_overall_score           65
invalid_metareview_score         0
invalid_recommendation           0
dtype: int64

In [42]:
validation["is_malformed"] = validation.any(axis=1)

In [43]:
validation["is_malformed"].sum()

248

In [44]:
(validation["is_malformed"] == (~valid_row)).all()

True

#### EXTRACT VALID ROWS

In [45]:
valid_df = df.loc[valid_row].copy()

In [46]:
valid_df.shape

(10273, 15)

In [47]:
valid_df.isnull().sum()

submission_id           0
paper_title             0
track                   0
author_ids              0
reviewer_id             0
assignment_timestamp    0
review_timestamp        0
soundness               0
excitement              0
confidence              0
overall_score           0
metareview_score        0
review_text             0
recommendation          0
line_number             0
dtype: int64

In [48]:
print("Total rows:", len(df))
print("Malformed rows:", validation["is_malformed"].sum())
print("Valid rows:", len(valid_df))

Total rows: 10521
Malformed rows: 248
Valid rows: 10273


#### SUMMARY STATISTICS

In [49]:
valid_df.dtypes

submission_id           object
paper_title             object
track                   object
author_ids              object
reviewer_id             object
assignment_timestamp    object
review_timestamp        object
soundness               object
excitement              object
confidence              object
overall_score           object
metareview_score        object
review_text             object
recommendation          object
line_number              int64
dtype: object

In [50]:
for col in ["soundness", "excitement", "confidence"]:
    valid_df[col] = pd.to_numeric(valid_df[col]).astype("int64")

for col in ["overall_score", "metareview_score"]:
    valid_df[col] = pd.to_numeric(valid_df[col]).astype("float64")

In [51]:
valid_df.dtypes

submission_id            object
paper_title              object
track                    object
author_ids               object
reviewer_id              object
assignment_timestamp     object
review_timestamp         object
soundness                 int64
excitement                int64
confidence                int64
overall_score           float64
metareview_score        float64
review_text              object
recommendation           object
line_number               int64
dtype: object

In [52]:
valid_review_count = len(valid_df)
valid_review_count

10273

In [53]:
papers_reviewed = valid_df["submission_id"].nunique()
papers_reviewed

3499

In [54]:
top_reviewers_count = (
    valid_df["reviewer_id"]
    .value_counts()
    .head(3)
)

top_reviewers = [
    [reviewer_id, int(count)]
    for reviewer_id, count in top_reviewers_count.items()
]

top_reviewers

[['R0817', 25], ['R0495', 23], ['R0668', 21]]

In [55]:
track_distribution = (
    valid_df["track"]
    .value_counts()
    .reindex(allowed_tracks, fill_value=0)
    .to_dict()
)

track_distribution = {
    track: int(count)
    for track, count in track_distribution.items()
}

track_distribution

{'MainConference': 5493, 'Findings': 2747, 'Demo': 483, 'Workshop': 1550}

In [56]:
recommendation_distribution = (
    valid_df["recommendation"]
    .value_counts()
    .reindex(
        allowed_recommendations,
        fill_value=0
    )
    .to_dict()
)

recommendation_distribution

{'Strong Reject': 266,
 'Borderline': 3740,
 'Accept': 3009,
 'Strong Accept': 352,
 'Reject': 2906}

In [57]:
numeric_columns = ["soundness", "excitement", "confidence", "overall_score"]

In [58]:
average_scores = {
    col: round(float(valid_df[col].mean()), 3)
    for col in numeric_columns
}

average_scores

{'soundness': 2.76,
 'excitement': 2.76,
 'confidence': 3.36,
 'overall_score': 5.512}

In [59]:
score_correlation_overall_vs_excitement = round(float(
    valid_df["overall_score"].corr(
        valid_df["excitement"]
    )
), 3)

score_correlation_overall_vs_excitement

0.778

#### ANOMALIES

In [60]:
anomalies = []

#### score_outlier

In [61]:
overall_mean = valid_df["overall_score"].mean()

In [62]:
overall_std = valid_df["overall_score"].std(ddof=0)

In [63]:
outlier_threshold = overall_mean + 3 * overall_std

In [64]:
print("Mean:", overall_mean)
print("Population std:", overall_std)
print("Threshold:", outlier_threshold)

Mean: 5.5117979168694635
Population std: 1.4957257670380573
Threshold: 9.998975217983634


In [65]:
score_outliers = valid_df[
    valid_df["overall_score"] > outlier_threshold
]

len(score_outliers)

13

#### coi_violation

In [66]:
coi_mask = valid_df.apply(
    lambda row: row["reviewer_id"] in row["author_ids"].split("|"),
    axis=1
)

coi_violations = valid_df[coi_mask]
print("COI violations:", len(coi_violations))

COI violations: 60


In [67]:
for _, row in coi_violations.iterrows():

    anomalies.append({
        "line_number": int(row["line_number"]),
        "submission_id": row["submission_id"],
        "reviewer_id": row["reviewer_id"],
        "type": "coi_violation",
        "reason": (
            f"Reviewer {row['reviewer_id']} "
            f"appears in the author list for "
            f"submission {row['submission_id']}"
        )
    })

#### metareview_inconsistency

In [68]:
review_counts = valid_df["submission_id"].value_counts()

multi_review_submissions = review_counts[
    review_counts >= 2
].index

multi_review_df = valid_df[
    valid_df["submission_id"].isin(multi_review_submissions)
]

metareview_modes = (
    multi_review_df
    .groupby("submission_id")["metareview_score"]
    .agg(lambda x: x.mode().iloc[0])
)

multi_review_df = multi_review_df.copy()

multi_review_df["metareview_consensus"] = (
    multi_review_df["submission_id"]
    .map(metareview_modes)
)

metareview_violations = multi_review_df[
    multi_review_df["metareview_score"]
    != multi_review_df["metareview_consensus"]
]

In [69]:
for _, row in metareview_violations.iterrows():

    anomalies.append({
        "line_number": int(row["line_number"]),
        "submission_id": row["submission_id"],
        "reviewer_id": row["reviewer_id"],
        "type": "metareview_inconsistency",
        "reason": (
            f"metareview_score {row['metareview_score']} "
            f"differs from the submission consensus "
            f"value {row['metareview_consensus']}"
        )
    })

#### duplicate_review

In [70]:
duplicate_mask = valid_df.duplicated(
    subset=["submission_id", "reviewer_id"],
    keep="first"
)

duplicate_reviews = valid_df[duplicate_mask]

In [71]:
for _, row in duplicate_reviews.iterrows():

    anomalies.append({
        "line_number": int(row["line_number"]),
        "submission_id": row["submission_id"],
        "reviewer_id": row["reviewer_id"],
        "type": "duplicate_review",
        "reason": (
            "The same submission_id and reviewer_id "
            "combination appeared previously in the valid rows."
        )
    })

#### recommendation_score_mismatch

In [72]:
recommendation = valid_df["recommendation"]
score = valid_df["overall_score"]

recommendation_valid = (
    ((recommendation == "Strong Reject") & (score >= 1.0) & (score < 2.5))
    |
    ((recommendation == "Reject") & (score >= 2.5) & (score < 5.0))
    |
    ((recommendation == "Borderline") & (score >= 5.0) & (score < 6.0))
    |
    ((recommendation == "Accept") & (score >= 6.0) & (score < 8.0))
    |
    ((recommendation == "Strong Accept") & (score >= 8.0) & (score <= 10.0))
)

recommendation_violations = valid_df[
    ~recommendation_valid
]

In [73]:
for _, row in recommendation_violations.iterrows():

    anomalies.append({
        "line_number": int(row["line_number"]),
        "submission_id": row["submission_id"],
        "reviewer_id": row["reviewer_id"],
        "type": "recommendation_score_mismatch",
        "reason": (
            f"Recommendation '{row['recommendation']}' "
            f"is inconsistent with overall_score "
            f"{row['overall_score']:.3f}"
        )
    })

#### confidence_score_mismatch

In [74]:
condition_1 = (
    (valid_df["confidence"] <= 2)
    &
    (
        (valid_df["overall_score"] < 3.0)
        |
        (valid_df["overall_score"] > 8.0)
    )
)

condition_2 = (
    (valid_df["confidence"] == 5)
    &
    (valid_df["overall_score"] >= 4.0)
    &
    (valid_df["overall_score"] <= 6.0)
)

confidence_violations = valid_df[
    condition_1 | condition_2
]

In [75]:
for _, row in confidence_violations.iterrows():

    anomalies.append({
        "line_number": int(row["line_number"]),
        "submission_id": row["submission_id"],
        "reviewer_id": row["reviewer_id"],
        "type": "confidence_score_mismatch",
        "reason": (
            f"confidence {row['confidence']} is inconsistent "
            f"with overall_score {row['overall_score']:.3f}"
        )
    })

#### review_before_assignment

In [76]:
valid_df["assignment_dt"] = pd.to_datetime(
    valid_df["assignment_timestamp"],
    utc=True
)

valid_df["review_dt"] = pd.to_datetime(
    valid_df["review_timestamp"],
    utc=True
)

review_before_assignment = valid_df[
    valid_df["review_dt"] < valid_df["assignment_dt"]
]

In [77]:
for _, row in review_before_assignment.iterrows():

    anomalies.append({
        "line_number": int(row["line_number"]),
        "submission_id": row["submission_id"],
        "reviewer_id": row["reviewer_id"],
        "type": "review_before_assignment",
        "reason": (
            "review_timestamp occurs before "
            "assignment_timestamp."
        )
    })

In [78]:
from collections import Counter

Counter(
    anomaly["type"]
    for anomaly in anomalies
)

Counter({'recommendation_score_mismatch': 4863,
         'confidence_score_mismatch': 959,
         'coi_violation': 60,
         'metareview_inconsistency': 50,
         'review_before_assignment': 49,
         'duplicate_review': 40})

#### BUILD JSON

In [79]:
results = {
    "valid_review_count": int(valid_review_count),

    "papers_reviewed": int(papers_reviewed),

    "top_reviewers": top_reviewers,

    "track_distribution": track_distribution,

    "recommendation_distribution": recommendation_distribution,

    "average_scores": average_scores,

    "score_correlation_overall_vs_excitement": score_correlation_overall_vs_excitement,

    "anomalies": anomalies
}

In [80]:
assert isinstance(results["valid_review_count"], int)

assert isinstance(results["papers_reviewed"], int)

assert len(results["top_reviewers"]) <= 3

assert set(results["track_distribution"]) == {
    "MainConference",
    "Findings",
    "Workshop",
    "Demo"
}

assert set(results["recommendation_distribution"]) == {
    "Strong Accept",
    "Accept",
    "Borderline",
    "Reject",
    "Strong Reject"
}

assert set(results["average_scores"]) == {
    "soundness",
    "excitement",
    "confidence",
    "overall_score"
}

assert isinstance(
    results["score_correlation_overall_vs_excitement"],
    float
)

assert isinstance(results["anomalies"], list)

In [81]:
import json

print(
    json.dumps(
        results,
        indent=2
    )
)

{
  "valid_review_count": 10273,
  "papers_reviewed": 3499,
  "top_reviewers": [
    [
      "R0817",
      25
    ],
    [
      "R0495",
      23
    ],
    [
      "R0668",
      21
    ]
  ],
  "track_distribution": {
    "MainConference": 5493,
    "Findings": 2747,
    "Demo": 483,
    "Workshop": 1550
  },
  "recommendation_distribution": {
    "Strong Reject": 266,
    "Borderline": 3740,
    "Accept": 3009,
    "Strong Accept": 352,
    "Reject": 2906
  },
  "average_scores": {
    "soundness": 2.76,
    "excitement": 2.76,
    "confidence": 3.36,
    "overall_score": 5.512
  },
  "score_correlation_overall_vs_excitement": 0.778,
  "anomalies": [
    {
      "line_number": 412,
      "submission_id": "S02487",
      "reviewer_id": "R0374",
      "type": "coi_violation",
      "reason": "Reviewer R0374 appears in the author list for submission S02487"
    },
    {
      "line_number": 419,
      "submission_id": "S03162",
      "reviewer_id": "R0345",
      "type": "coi_violati

In [82]:
with open("results.json", "w", encoding="utf-8") as f:
    json.dump(
        results,
        f,
        indent=2,
        ensure_ascii=False
    )

In [83]:
import os

os.path.exists("results.json")

True